# Full cache H/L — TRAIN10k + val5000, T4 ×2
**Chỉ tạo cache, không huấn luyện, không đánh giá caption.**
H: cắt bốn vùng ảnh gốc rồi resize224. L: resize toàn ảnh224 trước rồi cắt/phóng cùng bốn vùng.
Dùng cùng CLIP frozen và projection frozen của A; pool14×14→7×7 trước projection/ReLU.
Full ở đây là **15.000 ảnh đã khóa**, không phải toàn bộ COCO.

Thêm **mscoco-2014**, **pair-source-fullframe-ce** (checkpoint A), **visual-cache**.
Chọn **T4 ×2**, Internet ON, **Save & Run All**. Không cần API token.
Dự kiến ~6,02GB tensor; ~55 phút là ước lượng từ benchmark, chưa tính mọi chi phí xác minh/copy và không bảo đảm ETA.

30 shard ×500ảnh, mỗi shard có H và L. Chạy tiếp cùng session giữ nguyên OUTPUT.
Nếu session cũ mất, phải lưu folder thành dataset rồi thêm vào input, điền RESUME_FROM bên dưới.
Chỉ các shard hoàn tất và có hash hợp lệ được dùng lại; shard đang ghi dở làm lại tối đa500ảnh mỗi GPU.
Việc resume không cứu được output chưa lưu khi Kaggle xóa session.

Chỉ coi thành công khi có `cache_receipt.json` với `status=CACHE_COMPLETE` và 15.000ảnh.
**Giữ cả folder `native_detail_HL_15k` làm dataset**, không upload ZIP review thay cache.
Gửi `native_detail_HL_15k_review.zip` để kiểm tra. Review không chứa HDF5.

In [ ]:
print('NATIVE DETAIL FULL CACHE: START',flush=True)
from pathlib import Path
CHECKPOINT=Path('/kaggle/input/datasets/ducanh2403/pair-source-fullframe-ce/upt_hidden_caption_A/checkpoint.bin')
COCO_JSON=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
COCO_IMAGES=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/images')
VISUAL_CACHE=Path('/kaggle/input/datasets/ducanh2403/visual-cache')
OUTPUT=Path('/kaggle/working/native_detail_HL_15k')
RESUME_FROM=None  # Optional Path('/kaggle/input/datasets/OWNER/DATASET/native_detail_HL_15k')
for p in (CHECKPOINT,COCO_JSON):assert p.is_file(),p
for p in (COCO_IMAGES,VISUAL_CACHE):assert p.is_dir(),p
if RESUME_FROM is not None:
    RESUME_FROM=Path(RESUME_FROM)
    assert (RESUME_FROM/'cache_spec.json').is_file(),'Point directly to folder containing cache_spec.json.'


In [ ]:
import base64, hashlib, json, os, shutil, subprocess, sys, tempfile, zlib
os.environ.update(PYTHONUNBUFFERED='1', HDF5_USE_FILE_LOCKING='FALSE', PYTHONHASHSEED='42',
    CUBLAS_WORKSPACE_CONFIG=':4096:8', OMP_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2', TOKENIZERS_PARALLELISM='false', HF_HUB_DISABLE_XET='1')
# Isolate each bootstrap: embedded tracked overlays from a prior run are intentional edits.
REPO = Path(tempfile.mkdtemp(prefix='Image_Captioning-hidden-native-detail-full-cache-'))/'repo'
LOG = Path('/tmp/upt_hidden_native-detail-full-cache.log')
PROJECT_COMMIT = 'a2625edce0f4f554a36aec04354b158ce95b0a6f'
def run(args, cwd=None):
    args = list(map(str, args))
    print('\nRunning:', ' '.join(args), flush=True)
    with LOG.open('a', encoding='utf-8') as log:
        log.write('\nRunning: ' + ' '.join(args) + '\n'); log.flush()
        with subprocess.Popen(args, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                text=True, errors='replace', bufsize=1) as process:
            for line in process.stdout:
                print(line, end='', flush=True); log.write(line); log.flush()
            status = process.wait()
    if status:
        raise RuntimeError(f'Command failed ({status}). Saved log: {LOG}')
def checkout(url, path, revision):
    if not path.exists():
        run(['git','clone',url,path])
    assert (path/'.git').is_dir(), path
    assert not subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=path,text=True).strip(), 'Preserve tracked edits; use a fresh runtime path.'
    run(['git','fetch','--depth','1','origin',revision], cwd=path)
    run(['git','checkout','--detach',revision], cwd=path)
checkout('https://github.com/Supzxjee/Image_Captioning.git', REPO, PROJECT_COMMIT)
SOURCES = json.loads(zlib.decompress(base64.b64decode('...')).decode('utf-8'))
SOURCE_HASHES = {'cache_data.py': '6b890186a807f02d77006f027ee9fe7ce6818a6ba93ee367430e7b2be382feaa', 'run_full_cache.py': '732902f79f670ceb365b9214ed3a1f08829e17e5b371ada7f1f16d6f0a203b39', 'test_full_cache.py': 'f0cd467f68279197f6ecc9fbb0a0d04eb86619389f3ace9d73a3f4764da8ff31', 'tile_data.py': '47059a7eeca1b7fd3edfa7bfe2a52bbab4ea6b506bcf322bb6962d163c1512be', 'run_benchmark.py': 'd686723ca406bf797593a69e8854657fd2b65f6f299affd48097217a88a4d0f5', 'captioning/data.py': '9db178959dd9ed70312cfc2f58e11a0db28148d3328b2feea80786f8f2770463', 'captioning/visual_cache.py': 'adbcc9678fff499fc767cdbf04a4966186a40729f51c5bfb2525e79f9b6b5fe1', 'captioning/scst_runtime.py': '3452ccfbd43876dc6381b78d1229d1d6b05fe723e3863a04f31c4a784497f669'}
for name, source in SOURCES.items():
    assert hashlib.sha256(source.encode()).hexdigest() == SOURCE_HASHES[name], name
    destination = REPO/name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(source, encoding='utf-8')
print('Pinned runtime and hash-checked embedded source READY', flush=True)


In [ ]:
run([sys.executable,'-m','pip','install','-q','transformers==4.57.1','h5py','Pillow','numpy','pandas'])
import torch
assert torch.cuda.device_count()==2 and all('T4' in torch.cuda.get_device_name(r) for r in range(2)),'Select T4x2.'
run([sys.executable,'-u','test_full_cache.py'],cwd=REPO)
CONFIG=Path('/tmp/native_detail_full_cache_config.json')
cfg=dict(checkpoint=str(CHECKPOINT),dataset_json=str(COCO_JSON),images=str(COCO_IMAGES),visual_cache=str(VISUAL_CACHE),
    output=str(OUTPUT),resume_from=str(RESUME_FROM) if RESUME_FROM else None,
    groups_archive=str(REPO/'experiments/noun_evidence_proposal/data/noun_evidence_train10k_v1_dataset.zip'),
    implementation_sha256=SOURCE_HASHES,project_commit=PROJECT_COMMIT)
CONFIG.write_text(json.dumps(cfg,indent=2),encoding='utf-8')
try:
    run([sys.executable,'-u','run_full_cache.py','run','--config',CONFIG],cwd=REPO)
finally:
    if OUTPUT.is_dir():
        # Keep each notebook attempt without overwriting previous run logs/source.
        import uuid
        source_dir=OUTPUT/'notebook_attempts'/uuid.uuid4().hex;source_dir.mkdir(parents=True)
        shutil.copyfile(LOG,source_dir/'notebook.log')
        for name,source in SOURCES.items():
            destination=source_dir/name;destination.parent.mkdir(parents=True,exist_ok=True)
            destination.write_text(source,encoding='utf-8')
        run([sys.executable,'-u','run_full_cache.py','archive','--config',CONFIG],cwd=REPO)
receipt=json.loads((OUTPUT/'cache_receipt.json').read_text(encoding='utf-8'))
assert receipt['status']=='CACHE_COMPLETE' and receipt['images']==15000 and len(receipt['shards'])==30
print('KEEP ENTIRE FOLDER AS DATASET:',OUTPUT,flush=True)
print('SEND SMALL REVIEW ZIP:',str(OUTPUT)+'_review.zip',flush=True)
print('No training or caption evaluation has run.',flush=True)
